# SR3 Surface-Implied vs Realized Correlation Backtest (Listed-Only)

This notebook implements the strategy using existing repo infrastructure:
- `MDP/STIRFutures/STIRFutureMDP.py`
- `MDP/STIRFutures/STIRFutureOptionMDP.py`
- `BT/query_engine.py`, `BT/query_strategy.py`, `BT/query_actions.py`, `BT/triggers.py`, `BT/data_handler.py`

Trade expression is listed-only with single-contract SR3 options:
- Pair of ATM straddles on two SR3 underlyings (`CM` aliases)
- Vega-ratio sizing
- Event-centric entry/exit around FOMC

Notes:
- Current Barchart endpoints can rate-limit (`429`). The defaults below are conservative and include retry/throttling.
- The BT option handler expects singleton pricer objects; this notebook wraps the option MDP to flatten singleton lists.

In [1]:
import sys
sys.path.append("..")

import datetime as dt
import math
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql
import rateslib as rl
from tqdm.auto import tqdm

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements

from MDP.STIRFutures.STIRFutureMDP import STIRFutureMDP
from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from Query.IRSwaps._CENTRAL_BANK_DATES import _CENTRAL_BANK_DATES

from Query.STIRFutureOptions.STIRFutureOptionQuery import STIRFutureOptionQuery
from Query.STIRFutureOptions.STIRFutureOptionStructure import STIRFutureOptionStructure
from Query.STIRFutureOptions.STIRFutureOptionValue import STIRFutureOptionValue

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# -----------------------
# Strategy / run controls
# -----------------------
START_DATE = dt.date(2025, 1, 1)
END_DATE = dt.date(2026, 2, 27)

# Cross-tenor pair (front-ish vs belly-ish on the SR3 strip)
PAIR_CM = (2, 5)

# Calibration nodes for latent implied-corr fit (keep small to reduce API load)
CALIB_CM = (2, 3, 4, 5)

ENTRY_BDAYS_BEFORE_EVENT = 5
EXIT_BDAYS_AFTER_EVENT = 1

REALIZED_LOOKBACK = 20
SIGNAL_ABS_THRESHOLD = 0.10

EVENT_WINDOW_PRE_BDAYS = 0
EVENT_WINDOW_POST_BDAYS = 1

# Parsimonious latent-correlation model (single-factor loading slope)
KAPPA = 2.0
MIN_CALIB_POINTS = 3

# Execution / runtime safety
MAX_EVENTS = 8
MAX_TRADES = 6
REQUEST_SLEEP_SECONDS = 0.25
REQUEST_RETRIES = 3

POINT_VALUE = 2500.0  # 1 IMM index point = $2,500 for SR3 option/futures

# Optional heavy attribution pass (more API calls)
RUN_DELTA_HEDGE_ATTRIBUTION = False


In [ ]:
# Required MDPs from request
stirf_mdp = STIRFutureMDP(source="BARCHART_TOS_LIVE_STIRF-RL")
stirfo_mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")


class FlatOptionMDP:
    # Adapter for BT engine compatibility: flatten dict[str, list[pricer]] -> dict[str, pricer] when len==1.

    def __init__(self, inner):
        self.inner = inner

    @staticmethod
    def _flatten(out):
        if not isinstance(out, dict):
            return out
        flat = {}
        for k, v in out.items():
            if isinstance(v, list) and len(v) == 1:
                flat[k] = v[0]
            else:
                flat[k] = v
        return flat

    def get_pricer(self, request):
        return self._flatten(self.inner.get_pricer(request))

    def get_data(self, request):
        return self._flatten(self.inner.get_data(request))


flat_stirfo_mdp = FlatOptionMDP(stirfo_mdp)


In [ ]:
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)

ALL_BDAYS = [
    pd.Timestamp(x).date()
    for x in ql_cal_date_range(
        ql_cal=CAL,
        start=dt.datetime.combine(START_DATE, dt.time()),
        end=dt.datetime.combine(END_DATE, dt.time()),
    )
]
BDAY_TO_INDEX = {d: i for i, d in enumerate(ALL_BDAYS)}


def nearest_prev_bday(d: dt.date) -> Optional[dt.date]:
    if not ALL_BDAYS:
        return None
    x = d
    while x not in BDAY_TO_INDEX:
        x -= dt.timedelta(days=1)
        if x < ALL_BDAYS[0]:
            return None
    return x


def shift_bday(d: dt.date, offset: int) -> Optional[dt.date]:
    d0 = nearest_prev_bday(d)
    if d0 is None:
        return None
    i = BDAY_TO_INDEX[d0] + int(offset)
    if i < 0 or i >= len(ALL_BDAYS):
        return None
    return ALL_BDAYS[i]


def bdays_between(start: dt.date, end: dt.date) -> List[dt.date]:
    s = nearest_prev_bday(start)
    e = nearest_prev_bday(end)
    if s is None or e is None:
        return []
    i0 = BDAY_TO_INDEX[s]
    i1 = BDAY_TO_INDEX[e]
    if i1 < i0:
        return []
    return ALL_BDAYS[i0 : i1 + 1]


def _safe_first(v):
    if isinstance(v, list):
        return v[0] if v else None
    return v


def _retry_fetch(fetch_fn, retries=REQUEST_RETRIES):
    for i in range(retries):
        try:
            return fetch_fn()
        except Exception:
            if i == retries - 1:
                return {}
            time.sleep((i + 1) * 0.5)
    return {}


_option_cache: Dict[Tuple[dt.date, str], object] = {}
_future_cache: Dict[Tuple[dt.date, str], object] = {}


def get_option_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _option_cache]

    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirfo_mdp.get_data(
                {
                    "endpoint": "option_snapshot",
                    "symbols": missing,
                    "timestamp": as_of,
                    "show_tqdm": False,
                }
            )
        )
        for s in missing:
            _option_cache[(as_of, s)] = _safe_first(out.get(s))

    return {s: _option_cache.get((as_of, s)) for s in symbols}


def get_future_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _future_cache]

    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirf_mdp.get_data(
                {
                    "symbols": missing,
                    "timestamp": as_of,
                    "show_tqdm": False,
                }
            )
        )
        for s in missing:
            _future_cache[(as_of, s)] = _safe_first(out.get(s))

    return {s: _future_cache.get((as_of, s)) for s in symbols}


def option_snapshot_df(as_of: dt.date, symbols: List[str]) -> pd.DataFrame:
    pmap = get_option_pricers(as_of, symbols)
    rows = []
    for req, pr in pmap.items():
        if pr is None:
            continue
        rows.append(
            {
                "as_of": as_of,
                "request_symbol": req,
                "symbol": pr.symbol(),
                "underlying_symbol": pr.underlying_symbol(),
                "price": float(pr.price()) if np.isfinite(pr.price()) else np.nan,
                "delta": float(pr.delta()) if np.isfinite(pr.delta()) else np.nan,
                "vega": float(pr.vega()) if np.isfinite(pr.vega()) else np.nan,
                "iv_normal": float(pr.iv_normal()) if np.isfinite(pr.iv_normal()) else np.nan,
                "iv_normal_bps": float(pr.iv_normal_bps()) if np.isfinite(pr.iv_normal_bps()) else np.nan,
                "expiry_date": pr.expiry_date(),
            }
        )
    return pd.DataFrame(rows)


def underlying_tau_years(underlying_symbol: str, as_of: dt.date) -> float:
    code = str(underlying_symbol)[-3:]
    imm = rl.scheduling.get_imm(code=code)
    imm_date = dt.date(int(imm.year), int(imm.month), int(imm.day))
    return max((imm_date - as_of).days, 1) / 365.0


def fit_surface_implied_corr(vol_df: pd.DataFrame, pair_underlyings: Tuple[str, str], as_of: dt.date) -> Dict[str, float]:
    clean = vol_df.dropna(subset=["underlying_symbol", "iv_normal"]).copy()
    clean = clean.groupby("underlying_symbol", as_index=False).first()
    clean["tau"] = clean["underlying_symbol"].map(lambda s: underlying_tau_years(s, as_of))
    clean = clean[clean["tau"] > 0].copy()

    if len(clean) < MIN_CALIB_POINTS:
        return {"rho_imp": np.nan, "a": np.nan, "b": np.nan, "rmse": np.nan}

    x = clean["tau"].to_numpy(float)
    y = np.square(clean["iv_normal"].to_numpy(float))
    g = 1.0 - np.exp(-KAPPA * x)

    A = np.column_stack([g * g, np.ones_like(g)])
    params, *_ = np.linalg.lstsq(A, y, rcond=None)

    a = float(max(params[0], 1e-12))
    b = float(max(params[1], 1e-12))
    yhat = a * g * g + b
    rmse = float(np.sqrt(np.square(y - yhat).mean()))

    vmap = {}
    gmap = {}
    for _, row in clean.iterrows():
        gi = 1.0 - math.exp(-KAPPA * float(row["tau"]))
        vi = max(a * gi * gi + b, 1e-12)
        u = str(row["underlying_symbol"])
        vmap[u] = vi
        gmap[u] = gi

    u_i, u_j = pair_underlyings
    if u_i not in vmap or u_j not in vmap:
        rho_imp = np.nan
    else:
        numer = a * gmap[u_i] * gmap[u_j]
        denom = math.sqrt(vmap[u_i] * vmap[u_j])
        rho_imp = float(np.clip(numer / denom, -0.999, 0.999)) if denom > 0 else np.nan

    return {"rho_imp": rho_imp, "a": a, "b": b, "rmse": rmse}


def realized_corr_from_futures(sym_i: str, sym_j: str, end_date: dt.date, lookback: int) -> float:
    end_bday = nearest_prev_bday(end_date)
    if end_bday is None:
        return np.nan
    i_end = BDAY_TO_INDEX[end_bday]
    i_start = i_end - int(lookback)
    if i_start < 1:
        return np.nan

    dates = ALL_BDAYS[i_start - 1 : i_end + 1]
    rows = []
    for d in dates:
        pmap = get_future_pricers(d, [sym_i, sym_j])
        pi = pmap.get(sym_i)
        pj = pmap.get(sym_j)
        if pi is None or pj is None:
            continue
        rows.append((d, float(pi.price()), float(pj.price())))

    if len(rows) < max(10, lookback // 2):
        return np.nan

    df = pd.DataFrame(rows, columns=["date", "i", "j"]).set_index("date").sort_index()
    return float(df["i"].diff().corr(df["j"].diff()))


def event_realized_corr(sym_i: str, sym_j: str, event_date: dt.date, pre_bdays: int, post_bdays: int) -> float:
    d0 = nearest_prev_bday(event_date)
    if d0 is None:
        return np.nan

    i0 = BDAY_TO_INDEX[d0]
    i_start = max(0, i0 - int(pre_bdays))
    i_end = min(len(ALL_BDAYS) - 1, i0 + int(post_bdays))
    dates = ALL_BDAYS[i_start : i_end + 1]
    if len(dates) < 2:
        return np.nan

    rows = []
    for d in dates:
        pmap = get_future_pricers(d, [sym_i, sym_j])
        pi = pmap.get(sym_i)
        pj = pmap.get(sym_j)
        if pi is None or pj is None:
            continue
        rows.append((d, float(pi.price()), float(pj.price())))

    if len(rows) < 2:
        return np.nan

    df = pd.DataFrame(rows, columns=["date", "i", "j"]).set_index("date").sort_index()
    return float(df["i"].diff().corr(df["j"].diff()))


In [ ]:
# -----------------------
# Build event-level signals
# -----------------------
fomc_events = sorted(
    {
        v[0]
        for v in _CENTRAL_BANK_DATES["USD-FEDFUNDS"].values()
        if START_DATE <= v[0] <= END_DATE
    }
)
if MAX_EVENTS is not None:
    fomc_events = fomc_events[:MAX_EVENTS]

rows = []
for ev in tqdm(fomc_events, desc="Signal Build"):
    entry_date = shift_bday(ev, -ENTRY_BDAYS_BEFORE_EVENT)
    exit_date = shift_bday(ev, EXIT_BDAYS_AFTER_EVENT)
    if entry_date is None or exit_date is None or entry_date >= exit_date:
        continue

    calib_symbols = [f"SFRCM{k}|ATMS" for k in CALIB_CM]
    snap = option_snapshot_df(entry_date, calib_symbols)
    if snap.empty:
        continue

    req_i = f"SFRCM{PAIR_CM[0]}|ATMS"
    req_j = f"SFRCM{PAIR_CM[1]}|ATMS"
    ri = snap[snap["request_symbol"] == req_i]
    rj = snap[snap["request_symbol"] == req_j]
    if ri.empty or rj.empty:
        continue

    ri = ri.iloc[0]
    rj = rj.iloc[0]

    fit = fit_surface_implied_corr(
        snap,
        (str(ri["underlying_symbol"]), str(rj["underlying_symbol"])),
        entry_date,
    )
    rho_imp = float(fit["rho_imp"])

    rho_real_prior = realized_corr_from_futures(
        str(ri["underlying_symbol"]),
        str(rj["underlying_symbol"]),
        entry_date,
        REALIZED_LOOKBACK,
    )

    if (not np.isfinite(rho_imp)) or (not np.isfinite(rho_real_prior)):
        continue

    sigma_i = float(ri["iv_normal"])
    sigma_j = float(rj["iv_normal"])
    if (not np.isfinite(sigma_i)) or (not np.isfinite(sigma_j)) or sigma_i <= 0 or sigma_j <= 0:
        continue

    # Vega ratio: N_i / N_j ~ sigma_j / sigma_i
    ratio_i = sigma_j / sigma_i
    ratio_j = 1.0
    norm = max(abs(ratio_i), abs(ratio_j), 1e-12)
    ratio_i /= norm
    ratio_j /= norm

    rho_real_event = event_realized_corr(
        str(ri["underlying_symbol"]),
        str(rj["underlying_symbol"]),
        ev,
        EVENT_WINDOW_PRE_BDAYS,
        EVENT_WINDOW_POST_BDAYS,
    )

    rows.append(
        {
            "event_date": ev,
            "entry_date": entry_date,
            "exit_date": exit_date,
            "symbol_i": str(ri["symbol"]),
            "symbol_j": str(rj["symbol"]),
            "underlying_i": str(ri["underlying_symbol"]),
            "underlying_j": str(rj["underlying_symbol"]),
            "vol_i": sigma_i,
            "vol_j": sigma_j,
            "ratio_i": ratio_i,
            "ratio_j": ratio_j,
            "rho_imp": rho_imp,
            "rho_real_prior": float(rho_real_prior),
            "delta_rho": float(rho_imp - rho_real_prior),
            "rho_real_event": float(rho_real_event) if np.isfinite(rho_real_event) else np.nan,
            "fit_rmse": float(fit["rmse"]) if np.isfinite(fit["rmse"]) else np.nan,
        }
    )

signals_df = pd.DataFrame(rows)
if signals_df.empty:
    print("No valid signals were produced (likely data gaps or throttling).")
else:
    signals_df = signals_df.sort_values("entry_date").reset_index(drop=True)
    signals_df["signal_abs"] = signals_df["delta_rho"].abs()
    signals_df["trade_side"] = np.where(signals_df["delta_rho"] > 0.0, 1.0, -1.0)
    # delta_rho > 0 => sell corr => long both straddles
    # delta_rho < 0 => buy corr => short both straddles
    signals_df["weight_i"] = signals_df["trade_side"] * signals_df["ratio_i"]
    signals_df["weight_j"] = signals_df["trade_side"] * signals_df["ratio_j"]
    signals_df["tag"] = [f"sr3corr_{d:%Y%m%d}_{i:02d}" for i, d in enumerate(signals_df["entry_date"], 1)]

signals_df


In [ ]:
if signals_df.empty:
    trade_df = signals_df.copy()
else:
    trade_df = signals_df[signals_df["signal_abs"] >= SIGNAL_ABS_THRESHOLD].copy()
    if MAX_TRADES is not None:
        trade_df = trade_df.head(MAX_TRADES)

print(f"Signals: {len(signals_df)} | Trades selected: {len(trade_df)}")
trade_df[[
    "event_date", "entry_date", "exit_date", "symbol_i", "symbol_j",
    "rho_imp", "rho_real_prior", "delta_rho", "weight_i", "weight_j"
]] if not trade_df.empty else trade_df


In [ ]:
# -----------------------
# Build BT triggers/orders
# -----------------------
triggers = []

if not trade_df.empty:
    for r in trade_df.itertuples(index=False):
        q_i = STIRFutureOptionQuery(
            structure=STIRFutureOptionStructure.OUTRIGHT,
            value=STIRFutureOptionValue.PRICE,
            symbol=r.symbol_i,
            structure_kwargs={"symbol": r.symbol_i, "risk_weights": [float(r.weight_i)]},
            tags=(r.tag, "sr3_corr", "leg_i"),
        )
        q_j = STIRFutureOptionQuery(
            structure=STIRFutureOptionStructure.OUTRIGHT,
            value=STIRFutureOptionValue.PRICE,
            symbol=r.symbol_j,
            structure_kwargs={"symbol": r.symbol_j, "risk_weights": [float(r.weight_j)]},
            tags=(r.tag, "sr3_corr", "leg_j"),
        )

        enter = DateTrigger(
            DateTriggerRequirements(dates=[r.entry_date]),
            actions=[
                AddQueryAction(query=q_i, meta={"trade_tag": r.tag, "leg": "i"}),
                AddQueryAction(query=q_j, meta={"trade_tag": r.tag, "leg": "j"}),
            ],
        )
        exit_ = DateTrigger(
            DateTriggerRequirements(dates=[r.exit_date]),
            actions=[UnwindPositionsAction(match_tag=r.tag, fee=0.0)],
        )
        triggers.extend([enter, exit_])

print(f"Built triggers: {len(triggers)}")


In [ ]:
# -----------------------
# Run BT engine
# -----------------------
bt = None
mtm = pd.Series(dtype=float)

if triggers:
    tg = TimeGrid(
        ql_cal_date_range(
            ql_cal=CAL,
            start=dt.datetime.combine(START_DATE, dt.time()),
            end=dt.datetime.combine(END_DATE, dt.time()),
        )
    )

    strategy = QueryStrategy(
        name="SR3 Surface vs Realized Correlation",
        triggers=triggers,
        mdps={"STIRFUTUREOPTION": flat_stirfo_mdp},
    )

    bt = QueryDrivenBacktest(
        time_grid=tg,
        strategy=strategy,
        show_progress=True,
        progress_desc="BT SR3 Corr",
    )
    bt.run()

    mtm = pd.Series(bt.mtm_history).sort_index()
    print("Final MTM:", float(mtm.iloc[-1]) if len(mtm) else 0.0)
else:
    print("No triggers to run.")

mtm.tail()


In [ ]:
if len(mtm):
    plt.figure(figsize=(12, 5))
    plt.plot(mtm.index, mtm.values, lw=1.6)
    plt.title("SR3 Corr Strategy MTM")
    plt.ylabel("PnL ($)")
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# -----------------------
# Lightweight attribution
# -----------------------
# Uses entry/exit option marks for each trade (same symbols as BT positions).

attr_df = pd.DataFrame()

if not trade_df.empty:
    out_rows = []
    for r in tqdm(trade_df.to_dict(orient="records"), desc="Attribution"):
        o0 = get_option_pricers(r["entry_date"], [r["symbol_i"], r["symbol_j"]])
        o1 = get_option_pricers(r["exit_date"], [r["symbol_i"], r["symbol_j"]])

        p0i = o0.get(r["symbol_i"])
        p0j = o0.get(r["symbol_j"])
        p1i = o1.get(r["symbol_i"])
        p1j = o1.get(r["symbol_j"])

        if any(x is None for x in [p0i, p0j, p1i, p1j]):
            continue

        unhedged_pnl = POINT_VALUE * (
            float(r["weight_i"]) * (float(p1i.price()) - float(p0i.price()))
            + float(r["weight_j"]) * (float(p1j.price()) - float(p0j.price()))
        )

        out_rows.append(
            {
                **r,
                "unhedged_pnl": float(unhedged_pnl),
                "rho_gap_close": float(r.get("rho_real_event", np.nan) - r.get("rho_real_prior", np.nan))
                if np.isfinite(r.get("rho_real_event", np.nan)) and np.isfinite(r.get("rho_real_prior", np.nan))
                else np.nan,
            }
        )

    attr_df = pd.DataFrame(out_rows)

attr_df[[
    "entry_date", "exit_date", "symbol_i", "symbol_j", "delta_rho",
    "rho_real_prior", "rho_real_event", "unhedged_pnl"
]] if not attr_df.empty else attr_df


In [ ]:
if not attr_df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].scatter(attr_df["delta_rho"], attr_df["unhedged_pnl"])
    ax[0].axvline(0.0, color="k", lw=0.8)
    ax[0].axhline(0.0, color="k", lw=0.8)
    ax[0].set_title("P&L vs Signal (delta_rho)")
    ax[0].set_xlabel("delta_rho = rho_imp - rho_real_prior")
    ax[0].set_ylabel("Unhedged P&L ($)")

    valid = attr_df.dropna(subset=["rho_gap_close", "unhedged_pnl"])
    ax[1].scatter(valid["rho_gap_close"], valid["unhedged_pnl"])
    ax[1].axvline(0.0, color="k", lw=0.8)
    ax[1].axhline(0.0, color="k", lw=0.8)
    ax[1].set_title("P&L vs Event Corr Realization")
    ax[1].set_xlabel("rho_real_event - rho_real_prior")
    ax[1].set_ylabel("Unhedged P&L ($)")

    plt.tight_layout()
    plt.show()


In [ ]:
# -----------------------
# Optional: delta-hedge overlay (more API intensive)
# -----------------------

def trade_delta_hedge_pnl(row: pd.Series) -> float:
    dts = bdays_between(row["entry_date"], row["exit_date"])
    if len(dts) < 2:
        return 0.0

    hedge = 0.0
    for d0, d1 in zip(dts[:-1], dts[1:]):
        opt0 = get_option_pricers(d0, [row["symbol_i"], row["symbol_j"]])
        fut0 = get_future_pricers(d0, [row["underlying_i"], row["underlying_j"]])
        fut1 = get_future_pricers(d1, [row["underlying_i"], row["underlying_j"]])

        oi = opt0.get(row["symbol_i"])
        oj = opt0.get(row["symbol_j"])
        f0i = fut0.get(row["underlying_i"])
        f0j = fut0.get(row["underlying_j"])
        f1i = fut1.get(row["underlying_i"])
        f1j = fut1.get(row["underlying_j"])

        if any(x is None for x in [oi, oj, f0i, f0j, f1i, f1j]):
            continue

        di = float(oi.delta()) if np.isfinite(oi.delta()) else 0.0
        dj = float(oj.delta()) if np.isfinite(oj.delta()) else 0.0
        dfi = float(f1i.price() - f0i.price())
        dfj = float(f1j.price() - f0j.price())

        hedge += -POINT_VALUE * (
            float(row["weight_i"]) * di * dfi
            + float(row["weight_j"]) * dj * dfj
        )

    return float(hedge)


if RUN_DELTA_HEDGE_ATTRIBUTION and not attr_df.empty:
    attr_df = attr_df.copy()
    attr_df["delta_hedge_pnl"] = attr_df.apply(trade_delta_hedge_pnl, axis=1)
    attr_df["hedged_pnl"] = attr_df["unhedged_pnl"] + attr_df["delta_hedge_pnl"]
    display(attr_df[["entry_date", "delta_rho", "unhedged_pnl", "delta_hedge_pnl", "hedged_pnl"]])
else:
    print("Delta-hedge overlay skipped. Set RUN_DELTA_HEDGE_ATTRIBUTION=True to run.")


## Interpretation / Risk Notes

- `delta_rho > 0` path in this notebook is implemented as **long both straddles** (sell correlation exposure in the YCSO identity sense).
- `delta_rho < 0` path is **short both straddles** (buy correlation exposure).
- Implied correlation is latent and model-dependent (single-factor + fixed loading slope `KAPPA`).
- The pair is not a pure correlation swap; residual vol-ratio and smile effects remain.
- If you see sparse signals or empty trades, reduce request load (`MAX_EVENTS`, `MAX_TRADES`) and/or increase `REQUEST_SLEEP_SECONDS`.